In [ ]:
!pip install transformers torch

# **Bài 1: Khôi phục Masked Token (Masked Language Modeling)**
## **Code:**

In [ ]:
from transformers import pipeline
# 1. Tải pipeline "fill-mask"
# Pipeline này sẽ tự động tải một mô hình mặc định phù hợp (thường là một biến thể của BERT)
mask_filler = pipeline("fill-mask")
# 2. Câu đầu vào với token <mask>
input_sentence = "Hanoi is the <mask> of Vietnam."
# 3. Thực hiện dự đoán
# top_k=5 yêu cầu mô hình trả về 5 dự đoán hàng đầu
predictions = mask_filler(input_sentence, top_k=5)
# 4. In kết quả
print(f"Câu gốc: {input_sentence}")
for pred in predictions:
    print(f"Dự đoán: '{pred['token_str']}' với độ tin cậy: {pred['score']:.4f}")
    print(f" -> Câu hoàn chỉnh: {pred['sequence']}")

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8 (https://huggingface.co/distilbert/distilroberta-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Câu gốc: Hanoi is the <mask> of Vietnam.
Dự đoán: ' capital' với độ tin cậy: 0.9341
 -> Câu hoàn chỉnh: Hanoi is the capital of Vietnam.
Dự đoán: ' Republic' với độ tin cậy: 0.0300
 -> Câu hoàn chỉnh: Hanoi is the Republic of Vietnam.
Dự đoán: ' Capital' với độ tin cậy: 0.0105
 -> Câu hoàn chỉnh: Hanoi is the Capital of Vietnam.
Dự đoán: ' birthplace' với độ tin cậy: 0.0054
 -> Câu hoàn chỉnh: Hanoi is the birthplace of Vietnam.
Dự đoán: ' heart' với độ tin cậy: 0.0014
 -> Câu hoàn chỉnh: Hanoi is the heart of Vietnam.


## **Trả lời câu hỏi:**

### 1. Mô hình đã dự đoán đúng từ "capital" không?

Mô hình đã dự đoán đúng từ `capital` với độ tin cậy rất cao: **0.9341**.


### 2. Tại sao các mô hình Encoder-only như BERT lại phù hợp cho tác vụ này?

 Kết quả `Hanoi is the <mask> of Vietnam` $\rightarrow$ `capital` là một ví dụ sách giáo khoa để giải thích sức mạnh vượt trội của kiến trúc **Encoder-only**.

#### A. Hiểu ngữ cảnh hai chiều (Bidirectional Context)

Khác biệt lớn nhất so với các mô hình RNN/LSTM truyền thống là khả năng xử lý ngữ cảnh:

* **Hạn chế của mô hình đơn hướng:** Để điền đúng từ vào `<mask>`, mô hình không thể chỉ đọc từ trái sang phải (`Hanoi is the...`). Nếu chỉ đọc chiều này, mô hình có thể điền là `"city"`, `"pride"`, `"largest city"`, v.v.
* **Sức mạnh của Encoder:** Mô hình nhìn thấy **cả hai phía cùng lúc**: Nó thấy chủ ngữ là **"Hanoi"** **VÀ** bổ ngữ phía sau là **"of Vietnam"**.
* **Kết quả:** Sự kết hợp giữa "Hanoi" và "Vietnam" tạo ra một mối liên kết ngữ nghĩa mạnh mẽ nhất là quan hệ thủ đô - đất nước. Chính việc **"nhìn thấy tương lai"** (từ "Vietnam") đã giúp nó loại bỏ các từ sai và chọn đúng từ `capital`.

#### B. Masked Language Modeling (MLM)

Mô hình đã được huấn luyện chuyên biệt để làm nhiệm vụ này:

* **Mục tiêu huấn luyện:** Mô hình đã được huấn luyện bằng cách che đi (mask) hàng tỷ từ trong văn bản và buộc phải **đoán lại chúng** dựa trên ngữ cảnh xung quanh.
* **Tính phù hợp:** Bài toán điền từ vào chỗ trống là *sở trường* gốc của mô hình, giúp nó nhạy bén hơn bất kỳ mô hình RNN nào.
* **Lưu ý kỹ thuật:** Việc bạn thấy ký hiệu `<mask>` (thay vì `[MASK]`) và các từ có khoảng trắng phía trước (ví dụ `' capital'`) cho thấy bạn có thể đang sử dụng mô hình **RoBERTa** (một biến thể được tối ưu hóa hơn của BERT).

# **Bài 2: Dự đoán từ tiếp theo (Next Token Prediction)**
## **Code:**

In [ ]:
from transformers import pipeline
# 1. Tải pipeline "text-generation"
# Pipeline này sẽ tự động tải một mô hình phù hợp (thường là GPT-2)
generator = pipeline("text-generation")
# 2. Đoạn văn bản mồi
prompt = "The best thing about learning NLP is"
# 3. Sinh văn bản
# max_length: tổng độ dài của câu mồi và phần được sinh ra
# num_return_sequences: số lượng chuỗi kết quả muốn nhận
generated_texts = generator(prompt, max_length=50, num_return_sequences=1)
# 4. In kết quả
print(f"Câu mồi: '{prompt}'")
for text in generated_texts:
    print("Văn bản được sinh ra:")
    print(text['generated_text'])

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Câu mồi: 'The best thing about learning NLP is'
Văn bản được sinh ra:
The best thing about learning NLP is that it's an academic discipline in which the faculty members are very smart and creative, and the students are extremely interested in the field. The NLP faculty are very smart and creative.

So if you're an NLP learner, you're not going to teach the NLP course without some work. You're going to learn all the techniques that I would have worked on in my first semester, and you're going to learn all the techniques that I would have worked on in my second semester. How can you get this? No, you aren't going to teach the NLP!

This is something that you might not have even thought of. The problem with being a NLP student is that you're just not going to get the benefit of the NLP. How do you think you're going to get the benefit of the NLP?

I think that is a good question. I think that there are a lot of different ways to get back into the NLP. If you're going to study in the NLP, 

## **Trả lời câu hỏi:**

## 1. Kết quả sinh ra có hợp lý không?

Đoạn văn bản được sinh ra không hợp lý về mặt ngữ nghĩa và logic. Tuy có ngữ pháp đúng nhưng mắc các lỗi nghiêm trọng về cấu trúc và ý nghĩa (điển hình của các mô hình RNN/LSTM không được tinh chỉnh hoặc chiến thuật sinh văn bản tham lam):

| Vấn đề | Phân tích chi tiết |
| :--- | :--- |
| **Lặp lại (Repetition)** | Mô hình bị mắc kẹt trong việc lặp lại các cụm từ hoặc ý tưởng gần kề. Ví dụ: *"The NLP faculty are very smart and creative."* (lặp lại ý câu trước) và lặp lại việc nhắc đến *"go back to school"* nhiều lần. |
| **Mâu thuẫn logic** | Mô hình tự mâu thuẫn trong lập luận. Ví dụ: Dù câu mồi bắt đầu bằng ý tích cực (*"The best thing..."*), mô hình lại sinh ra: *"The problem with being a NLP student is that you're just **not going to get the benefit of the NLP**."* (phủ định ý chính). |
| **Trôi chủ đề (Topic Drift)** | Mô hình mất khả năng duy trì ngữ cảnh dài. Nó đang nói về việc học NLP, sau đó đột ngột chuyển sang các ngành khác như *"law school"* (trường luật) mà không có sự chuyển tiếp hợp lý. |

---

## 2. Tại sao các mô hình Decoder-only như GPT lại phù hợp cho tác vụ này?

Tác vụ này là **Sinh văn bản tự do (Open-ended Text Generation)**, và các mô hình Decoder-only (như GPT, GPT-2, GPT-3) là kiến trúc tối ưu nhất nhờ vào đặc tính **tự hồi quy** và kiến trúc **Transformer**:

### A. Sinh văn bản Tự hồi quy (Autoregressive Generation)
* Mô hình Decoder được thiết kế để dự đoán token tiếp theo ($\text{token}$) dựa trên tất cả các token đã được sinh ra trước đó.
* Quá trình này lặp đi lặp lại cho đến khi sinh ra đủ độ dài hoặc gặp token kết thúc chuỗi ($\text{<EOS>}$). Đây là cơ chế cơ bản để tạo ra văn bản tuần tự và mạch lạc.

### B. Cơ chế Masked Self-Attention
* Trong kiến trúc Decoder, cơ chế **Self-Attention** được "che mặt" (Masked). Điều này đảm bảo rằng khi mô hình đang dự đoán một từ, nó **chỉ nhìn vào các từ đã xuất hiện** (phía bên trái), chứ không nhìn vào các từ "tương lai" chưa được sinh ra.
* Đây là điều kiện bắt buộc và lý tưởng cho tác vụ sinh văn bản (chúng ta không thể biết trước kết quả).

### C. Ngữ cảnh dài và Tính mạch lạc
* Nhờ sử dụng kiến trúc **Transformer**, mô hình Decoder-only có thể xử lý và duy trì thông tin ngữ cảnh qua **hàng ngàn token** (độ dài cửa sổ ngữ cảnh), khắc phục hoàn toàn vấn đề **Trôi chủ đề** và **Lặp lại** của các mô hình RNN/LSTM khi sinh văn bản dài.
* Việc huấn luyện trên dữ liệu khổng lồ (Pre-training) giúp GPT học được logic, cấu trúc tường thuật, và cách đặt câu hỏi/trả lời một cách tự nhiên và có ý nghĩa.

# **Bài 3: Tính toán Vector biểu diễn của câu (Sentence Representation)**
## **Code:**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Chọn một mô hình BERT
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
# 2. Câu đầu vào
sentences = ["This is a sample sentence."]
# 3. Tokenize câu
# padding=True: đệm các câu ngắn hơn để có cùng độ dài
# truncation=True: cắt các câu dài hơn
# return_tensors='pt': trả về kết quả dưới dạng PyTorch tensors
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
# 4. Đưa qua mô hình để lấy hidden states
# torch.no_grad() để không tính toán gradient, tiết kiệm bộ nhớ
with torch.no_grad():
    outputs = model(**inputs)
# outputs.last_hidden_state chứa vector đầu ra của tất cả các token
last_hidden_state = outputs.last_hidden_state
# shape: (batch_size, sequence_length, hidden_size)
# 5. Thực hiện Mean Pooling
# Để tính trung bình chính xác, chúng ta cần bỏ qua các token đệm (padding tokens)
attention_mask = inputs['attention_mask']
mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
sentence_embedding = sum_embeddings / sum_mask
# 6. In kết quả
print("Vector biểu diễn của câu:")
print(sentence_embedding)
print("\nKích thước của vector:", sentence_embedding.shape)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Vector biểu diễn của câu:
tensor([[-6.3875e-02, -4.2837e-01, -6.6779e-02, -3.8430e-01, -6.5785e-02,
         -2.1826e-01,  4.7636e-01,  4.8659e-01,  3.9689e-05, -7.4274e-02,
         -7.4740e-02, -4.7635e-01, -1.9773e-01,  2.4824e-01, -1.2162e-01,
          1.6678e-01,  2.1045e-01, -1.4576e-01,  1.2637e-01,  1.8635e-02,
          2.4640e-01,  5.7090e-01, -4.7014e-01,  1.3782e-01,  7.3650e-01,
         -3.3808e-01, -5.0329e-02, -1.6453e-01, -4.3517e-01, -1.2900e-01,
          1.6516e-01,  3.4004e-01, -1.4930e-01,  2.2422e-02, -1.0488e-01,
         -5.1916e-01,  3.2964e-01, -2.2162e-01, -3.4206e-01,  1.1993e-01,
         -7.0148e-01, -2.3126e-01,  1.1224e-01,  1.2550e-01, -2.5191e-01,
         -4.6374e-01, -2.7261e-02, -2.8415e-01, -9.9250e-02, -3.7018e-02,
         -8.9192e-01,  2.5005e-01,  1.5816e-01,  2.2701e-01, -2.8497e-01,
          4.5300e-01,  5.0921e-03, -7.9441e-01, -3.1008e-01, -1.7403e-01,
          4.3029e-01,  1.6816e-01,  1.0590e-01, -4.8987e-01,  3.1856e-01,
          3.

## **Trả lời câu hỏi:**

## 1. Kích thước (chiều) của vector biểu diễn là bao nhiêu? Tương ứng tham số nào?

* **Kích thước:** Chiều của vector là **768**.
    * Bạn có thể thấy điều này trong `torch.Size([1, 768])`, trong đó `1` là batch size và `768` là số chiều đặc trưng.
* **Tham số tương ứng:** Con số này tương ứng với tham số **`hidden_size`** (kích thước lớp ẩn) trong kiến trúc của mô hình **BERT Base**.
    * Mô hình BERT Base có cấu trúc: 12 lớp (layers), 12 đầu attention (heads), và **768 đơn vị ẩn (hidden units)**.


## 2. Tại sao cần sử dụng `attention_mask` khi thực hiện Mean Pooling?

Mục đích chính: Để **loại bỏ ảnh hưởng của các token đệm (padding tokens)**, đảm bảo tính chính xác của vector đại diện.

**Giải thích chi tiết:**
1.  **Vấn đề Padding:** Khi xử lý theo batch, các câu ngắn phải thêm token `[PAD]` để có cùng độ dài với câu dài nhất.
2.  **Nếu không có Mask:** Việc tính trung bình cộng (Mean) cả các vector của token `[PAD]` sẽ làm vector đại diện câu bị sai lệch và "loãng" thông tin (vì token `[PAD]` không mang nghĩa).
3.  **Vai trò của `attention_mask`:**
    * Mask chứa giá trị **1** cho token thật và **0** cho token padding.
    * Khi tính toán, ta chỉ tổng hợp các vector có mask là 1 và chia cho tổng số lượng token thật (thay vì chia cho tổng độ dài chuỗi).